In [1]:
# Pydantic is a Python library used for data validation and parsing using Python type hints.
# It is commonly used to validate API inputs, request/response data, configuration settings, 
# and structured data, especially with FastAPI.

In [2]:
# ! pip install --quiet pydantic

### Why Pydantic

In [3]:
# assume that you have a table called paitents.db which has 2 columns name (str) and age (int)

In [4]:
def insert_paitent_data(name, age):
    print(name)
    print(age)
    print('Record added')

# user / programmer knows that the fn requires name and age needs to be passed ! 
# but he does not what datatype should be passed for age....
insert_paitent_data('Ray', 'Thirty')

Ray
Thirty
Record added


In [5]:
def insert_paitent_data(name, age):
    if type(name) == str and type(age) == int:
        print(name)
        print(age)
        print('Record added')
    else:
        print('record cannot be added. type mismatch')

insert_paitent_data('Amul', 30)
# So the typecasting issue can be sorted using the above code but its not scalable...
# Issue 1: 
# If we have other fns like update / delete paitent data the same need to be updated with this code.
# Also this a hardcoding system which many customers dont like in production

# Issue 2:
# If I want to give a condition that age has to be greater than 20 only ... additional conditions need to be given.

Amul
30
Record added


In [6]:
# Mainly customer does not like harcoding and scaling such conditions across multiple programs
# is an issue. There is where pydantic comes into picture !

### How Pydantic works !

In [7]:
from pydantic import BaseModel

In [8]:
class Person(BaseModel):
    # define the data schema
    name : str
    age : int
    city : str

person1 = Person(name='Kuldeep', age=45, city='Pune')
person1

Person(name='Kuldeep', age=45, city='Pune')

In [9]:
person1.age

45

In [10]:
import sys
try:
    person2 = Person(name='Kuldeep', age='forty-five', city='Pune')
    print(person2)
except:
    print(sys.exc_info())

(<class 'pydantic_core._pydantic_core.ValidationError'>, 1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='forty-five', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing, <traceback object at 0x1061f3680>)


In [11]:
from pydantic import ValidationError

try:
    person2 = Person(name='Kuldeep', age='forty-five', city='Pune')
    print(person2)
except ValidationError as ex:
    exceptions = ex

exceptions.errors()

[{'type': 'int_parsing',
  'loc': ('age',),
  'msg': 'Input should be a valid integer, unable to parse string as an integer',
  'input': 'forty-five',
  'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}]

### Required and Optional Fields

In [12]:
from pydantic import BaseModel
from typing import Optional
from pydantic import ValidationError

class Employee(BaseModel):
    id : int
    name : str
    department : str
    salary : Optional[float] = None
    is_active : Optional[bool] = None

try:
    emp1 = Employee(id=1, name='John', department='IT')
    print(emp1)
except ValidationError as ex:
    exceptions = ex
    print(exceptions.errors())

id=1 name='John' department='IT' salary=None is_active=None


In [13]:
emp2 = Employee(id=2, name='Jane', department='HR', salary=60000.00, is_active=False)
emp2

Employee(id=2, name='Jane', department='HR', salary=60000.0, is_active=False)

In [14]:
try:
    emp3 = Employee(id=2, name='Jane', department=3, salary=60000.00, is_active=False)
    print(emp3)
except ValidationError as ex:
    exceptions = ex
    print(exceptions.errors())  

[{'type': 'string_type', 'loc': ('department',), 'msg': 'Input should be a valid string', 'input': 3, 'url': 'https://errors.pydantic.dev/2.13/v/string_type'}]


### Nullable Fields

In [16]:
class Person(BaseModel):
    first_name : str | None = None
    last_name : str
    age : int = 0

p1 = Person(last_name='Smith', age = 10)
p1

Person(first_name=None, last_name='Smith', age=10)

In [17]:
Person.model_fields

{'first_name': FieldInfo(annotation=Union[str, NoneType], required=False, default=None),
 'last_name': FieldInfo(annotation=str, required=True),
 'age': FieldInfo(annotation=int, required=False, default=0)}

### Conditions on a Field

In [21]:
from pydantic import BaseModel, Field

class Item(BaseModel):
    name : str = Field(min_length=2, max_length=50)
    price : float = Field(gt = 0, le = 1000)
    quantity : int = Field(ge = 0)

# Valid instance
try:
    item = Item(name='Book', price=119.99, quantity=10)
    print(item)
except ValueError as e:
    print(e)

name='Book' price=119.99 quantity=10


### Using Pydantic on Sqlite

In [22]:
# SQLite3 is a fast, self-contained, and serverless SQL database engine that stores entire databases 
# in single, lightweight files. 

In [ ]:
# To create the DB use the below code

# import sqlite3

# # Connect to database
# conn = sqlite3.connect("users.db")

# # Create cursor
# cursor = conn.cursor()

# # Create users table
# cursor.execute("""
#     CREATE TABLE IF NOT EXISTS users (
#         id INTEGER PRIMARY KEY AUTOINCREMENT,
#         name TEXT NOT NULL,
#         age INTEGER,
#         email TEXT
#     )
# """)

# # Save changes
# conn.commit()

# # Close connection
# conn.close()

# print("Database and users table created successfully!")

In [ ]:
# to add entries in the db
# import sqlite3

# # Connect to database
# conn = sqlite3.connect("users.db")
# cursor = conn.cursor()

# # Sample users
# users = [
#     ("John", 25, "john@example.com"),
#     ("Priya", 30, "priya@example.com"),
#     ("Rahul", 28, "rahul@example.com"),
#     ("Sneha", 24, "sneha@example.com"),
#     ("Amit", 35, "amit@example.com")
# ]

# # Insert users
# cursor.executemany("""
#     INSERT INTO users (name, age, email)
#     VALUES (?, ?, ?)
# """, users)

# # Save changes
# conn.commit()

# # Close connection
# conn.close()

# print("Sample users added successfully!")

In [23]:
import sqlite3

# ==============================
# Connect to SQLite Database
# (Creates a file 'users.db' if it doesn't exist)
# ==============================
conn = sqlite3.connect("users.db")
cursor = conn.cursor()

# ==============================
# Create table for users (if not already present)
# ==============================
cursor.execute('''SELECT * FROM users''')

all_rows = cursor.fetchall()
for row in all_rows:
    print(row)
            
conn.close()

(1, 'Alice', 25, 'alice@example.com')
(2, 'Kuldeep', 45, 'kuldeep@example.com')
(4, 'Veena', 42, 'veena@example.com')
(5, 'Ray', 32, 'r@example.com')


In [25]:
from pydantic import BaseModel
class Employee(BaseModel):
    # define the data schema
    name : str
    age : int = Field(ge = 21, le = 35)
    email : str

# Valid instance
try:
    n = input('Enter Emp name: ')
    a = input('Enter Emp age: ')
    e = input('Enter Emp email: ')
    print(type(n), type(a), type(e))
    emp1 = Employee(name = n, age = a, email = e)
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    cursor.execute( "INSERT INTO users (name, age, email) VALUES (?, ?, ?)",
            (emp1.name, emp1.age, emp1.email))
    conn.commit()
    
    cursor.execute('''SELECT * FROM users''')
    all_rows = cursor.fetchall()
    for row in all_rows:
        print(row)
            
    conn.close()
except ValueError as e:
    print(e)

Enter Emp name:  Raju
Enter Emp age:  20
Enter Emp email:  raju@example.com


<class 'str'> <class 'str'> <class 'str'>
1 validation error for Employee
age
  Input should be greater than or equal to 21 [type=greater_than_equal, input_value='20', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal
